# Argus Backtest Analysis

Load and visualise results from `scripts/run_backtest.py`.

**Prerequisites:** Run the backtest first:
```bash
python scripts/run_backtest.py --ticker AAPL --start 2020-01-01 --end 2024-12-31
```

In [ ]:
import sys
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import mlflow
import numpy as np
import pandas as pd

# Add project root so src/ imports work from the notebook
sys.path.insert(0, str(Path.cwd().parent))

# ── Configuration — edit these to match your backtest run ───────────────── #
TICKER        = "AAPL"
START         = "2020-01-01"
END           = "2024-12-31"
INITIAL_CASH  = 100_000.0
RESULTS_DIR   = Path("../data/results")
MLFLOW_DB     = Path("../data/mlflow.db")
EXPERIMENT    = "argus_backtests"

tag = f"{TICKER}_{START}_{END}"
print(f"Analysing backtest: {tag}")

## 1. Load Equity Curve

In [ ]:
equity_csv = RESULTS_DIR / f"equity_{tag}.csv"

if not equity_csv.exists():
    raise FileNotFoundError(
        f"{equity_csv} not found.\n"
        f"Run: python scripts/run_backtest.py --ticker {TICKER} --start {START} --end {END}"
    )

portfolio_df = pd.read_csv(equity_csv, index_col=0, parse_dates=True)
print("Portfolio columns:", portfolio_df.columns.tolist())
print(f"Rows: {len(portfolio_df)}")
portfolio_df.head()

## 2. Load Trades

In [ ]:
trades_csv = RESULTS_DIR / f"trades_{tag}.csv"
trades_df = pd.read_csv(trades_csv, index_col=0, parse_dates=True) if trades_csv.exists() else pd.DataFrame()
print(f"Trades: {len(trades_df)}")
trades_df.head()

## 3. Build Buy-and-Hold Benchmark

In [ ]:
from src.data.market_data import MarketDataProvider

mdp = MarketDataProvider(cache_dir="../data/raw")
price_df = mdp.fetch_ohlcv(TICKER, start=START, end=END)

first_close = float(price_df["close"].iloc[0])
bnh_shares  = INITIAL_CASH / first_close
bnh_equity  = price_df["close"] * bnh_shares
bnh_equity.name = "bnh_equity"

print(f"Buy & hold: {bnh_shares:.2f} shares @ ${first_close:.2f}")
print(f"Final value: ${float(bnh_equity.iloc[-1]):,.0f}  ({(bnh_equity.iloc[-1]/INITIAL_CASH-1)*100:.1f}%)")

## 4. Equity Curve — Strategy vs Buy & Hold

In [ ]:
# Identify the equity column from PyBroker's portfolio DataFrame
equity_col = next(
    (c for c in portfolio_df.columns if "equity" in c.lower()),
    portfolio_df.columns[0],
)
equity = portfolio_df[equity_col]

fig, axes = plt.subplots(
    2, 1, figsize=(14, 8), sharex=True,
    gridspec_kw={"height_ratios": [3, 1]},
)

# ── Top panel: equity curves ─────────────────────────────────────────────── #
ax1 = axes[0]
ax1.plot(equity.index, equity.values, label="Argus Strategy", color="#2196F3", linewidth=1.5)
ax1.plot(bnh_equity.index, bnh_equity.values, label="Buy & Hold",
         color="#FF9800", linewidth=1.5, linestyle="--")
ax1.fill_between(equity.index, INITIAL_CASH, equity.values, alpha=0.08, color="#2196F3")
ax1.axhline(INITIAL_CASH, color="grey", linewidth=0.5, linestyle=":")
ax1.set_ylabel("Portfolio Value ($)")
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax1.legend()
ax1.set_title(f"Argus Strategy vs Buy & Hold — {TICKER} {START} to {END}")
ax1.grid(alpha=0.3)

# ── Bottom panel: strategy drawdown ─────────────────────────────────────── #
rolling_max = equity.cummax()
drawdown = (equity - rolling_max) / rolling_max * 100.0

ax2 = axes[1]
ax2.fill_between(drawdown.index, drawdown.values, 0, color="#F44336", alpha=0.5)
ax2.axhline(0, color="black", linewidth=0.5)
ax2.set_ylabel("Drawdown (%)")
ax2.set_xlabel("Date")
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax2.grid(alpha=0.3)

plt.tight_layout()

out_png = RESULTS_DIR / f"backtest_{tag}.png"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(out_png, dpi=150)
print(f"Saved: {out_png}")
plt.show()

## 5. Metrics from MLflow

In [ ]:

if not MLFLOW_DB.exists():
    print("MLflow DB not found — skipping metrics table.")
else:
    db_uri = f"sqlite:///{MLFLOW_DB.resolve()}"
    mlflow.set_tracking_uri(db_uri)

    runs = mlflow.search_runs(
        experiment_names=[EXPERIMENT],
        filter_string=f"params.ticker = '{TICKER}' AND params.start = '{START}'",
        order_by=["start_time DESC"],
        max_results=5,
    )

    if runs.empty:
        print(f"No runs found for {TICKER} {START}. Run the backtest first.")
    else:
        display_cols = [
            c for c in runs.columns
            if c.startswith("metrics.") or c in ["run_id", "start_time"]
        ]
        print(f"Found {len(runs)} run(s). Showing most recent:")
        latest = runs.iloc[0]

        metric_map = {
            "Sharpe Ratio":          latest.get("metrics.sharpe_ratio", float("nan")),
            "Sortino Ratio":         latest.get("metrics.sortino", float("nan")),
            "Calmar Ratio":          latest.get("metrics.calmar", float("nan")),
            "Max Drawdown %":        latest.get("metrics.max_drawdown_pct", float("nan")),
            "Win Rate":              latest.get("metrics.win_rate", float("nan")),
            "Profit Factor":         latest.get("metrics.profit_factor", float("nan")),
            "Total Return %":        latest.get("metrics.total_return_pct", float("nan")),
            "Annual Return %":       latest.get("metrics.annual_return_pct", float("nan")),
            "Annual Volatility %":   latest.get("metrics.annual_volatility_pct", float("nan")),
            "Trade Count":           latest.get("metrics.trade_count", float("nan")),
            "B&H Sharpe":            latest.get("metrics.bnh_sharpe", float("nan")),
            "B&H Total Return %":    latest.get("metrics.bnh_total_return_pct", float("nan")),
            "B&H Max Drawdown %":    latest.get("metrics.bnh_max_drawdown_pct", float("nan")),
            "Sharpe vs B&H":         latest.get("metrics.sharpe_vs_bnh", float("nan")),
            "Return vs B&H %":       latest.get("metrics.return_vs_bnh_pct", float("nan")),
        }

        metrics_series = pd.Series(metric_map, name="Value")
        display(metrics_series.to_frame())

## 6. Monthly Returns Heatmap

In [ ]:
# Resample equity to month-end
monthly_equity = equity.resample("ME").last()
monthly_ret = monthly_equity.pct_change().dropna() * 100.0

pivot = pd.DataFrame({
    "return": monthly_ret.values,
    "year":   monthly_ret.index.year,
    "month":  monthly_ret.index.month,
}, index=monthly_ret.index)

heatmap_data = pivot.pivot(index="year", columns="month", values="return")

# Symmetric colour scale around 0
abs_max = max(abs(heatmap_data.values[~np.isnan(heatmap_data.values)].max()), 1.0)
norm = mcolors.TwoSlopeNorm(vmin=-abs_max, vcenter=0.0, vmax=abs_max)

month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

fig, ax = plt.subplots(figsize=(14, max(3, len(heatmap_data) * 0.6)))
im = ax.imshow(heatmap_data.values, aspect="auto", cmap="RdYlGn", norm=norm)
plt.colorbar(im, ax=ax, label="Monthly Return (%)")

ax.set_xticks(range(12))
ax.set_xticklabels(month_labels)
ax.set_yticks(range(len(heatmap_data.index)))
ax.set_yticklabels(heatmap_data.index)

# Annotate each cell with the return value
for i, year in enumerate(heatmap_data.index):
    for j, month in enumerate(range(1, 13)):
        val = heatmap_data.loc[year, month] if month in heatmap_data.columns else float("nan")
        if not np.isnan(val):
            ax.text(j, i, f"{val:.1f}", ha="center", va="center", fontsize=7,
                    color="black" if abs(val) < abs_max * 0.7 else "white")

ax.set_title(f"Monthly Returns Heatmap — {TICKER} {START} to {END}")
plt.tight_layout()
plt.show()

## 7. Trade Distribution

In [ ]:
if trades_df.empty:
    print("No trades to analyse.")
else:
    # Find P&L column (PyBroker names it 'pnl')
    pnl_col = next((c for c in trades_df.columns if "pnl" in c.lower()), None)

    if pnl_col is None:
        print("No PnL column found. Columns:", trades_df.columns.tolist())
    else:
        pnl = trades_df[pnl_col].dropna()
        winners = pnl[pnl > 0]
        losers  = pnl[pnl < 0]

        fig, axes = plt.subplots(1, 2, figsize=(14, 4))

        # Left: P&L histogram
        ax = axes[0]
        ax.hist(winners, bins=20, color="#4CAF50", alpha=0.7, label=f"Winners ({len(winners)})")
        ax.hist(losers,  bins=20, color="#F44336", alpha=0.7, label=f"Losers ({len(losers)})")
        ax.axvline(0, color="black", linewidth=0.8)
        ax.set_xlabel("P&L ($)")
        ax.set_ylabel("Count")
        ax.set_title("Trade P&L Distribution")
        ax.legend()
        ax.grid(alpha=0.3)

        # Right: cumulative P&L
        ax2 = axes[1]
        cum_pnl = pnl.cumsum()
        ax2.plot(range(len(cum_pnl)), cum_pnl.values, color="#2196F3")
        ax2.axhline(0, color="grey", linewidth=0.5, linestyle=":")
        ax2.set_xlabel("Trade #")
        ax2.set_ylabel("Cumulative P&L ($)")
        ax2.set_title("Cumulative P&L per Trade")
        ax2.grid(alpha=0.3)

        plt.tight_layout()
        plt.show()

        print("\nTrade summary:")
        print(f"  Total trades : {len(pnl)}")
        print(f"  Winners      : {len(winners)} ({len(winners)/len(pnl)*100:.1f}%)")
        print(f"  Losers       : {len(losers)} ({len(losers)/len(pnl)*100:.1f}%)")
        print(f"  Avg win      : ${winners.mean():.2f}" if len(winners) else "  Avg win: N/A")
        print(f"  Avg loss     : ${losers.mean():.2f}" if len(losers)  else "  Avg loss: N/A")
        print(f"  Largest win  : ${pnl.max():.2f}")
        print(f"  Largest loss : ${pnl.min():.2f}")